# Tutorial 6: SBI Surrogate Learning (`sbi_npe`)

Estimated time: 30-50 minutes

## Prerequisites
`torch` and `sbi` installed. Run `mm doctor` to verify.

## Learning aims
- Fit/evaluate an SBI backend with the same user-facing surrogate interface
- Understand likelihood-free neural posterior estimation from simulation pairs
- Train a multi-output joint surrogate using a D-dim density estimator


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Ensure training dataset exists

In [ ]:
run_cli('run', 'tutorials/specs/model.toy.grid.json')


## Step 2: Fit/evaluate single-output SBI surrogate

In [ ]:
run_cli('surrogate', 'fit', 'tutorials/specs/surrogate.toy.sbi_npe.json')
import json
from metamodeler.spec import SurrogateSpec
from metamodeler.surrogates import eval_surrogate

spec_payload = json.loads(
    (ROOT / 'tutorials/specs/surrogate.toy.sbi_npe.json').read_text()
)
spec = SurrogateSpec.model_validate(spec_payload)
result = eval_surrogate(
    spec=spec,
    inputs_payload={'a': [0.25, 0.75, 1.25, 1.75], 'b': [0.2, 0.6, 1.0, 1.4]},
    n=200,
)
print('sample shape:', result['sample_shape'])


## Step 3: Plot SBI predictive summary (graphic)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

out_name = spec.outputs[0]
mean = np.asarray(result['summary']['mean'][out_name], dtype=float)
std = np.asarray(result['summary']['std'][out_name], dtype=float)
x = np.arange(len(mean))

plt.figure(figsize=(6, 4))
plt.errorbar(x, mean, yerr=std, fmt='o-', capsize=4, color='tab:orange')
plt.title(f'SBI surrogate predictive {out_name}: mean +/- std')
plt.xlabel('query point index')
plt.ylabel(f'predicted {out_name}')
plt.grid(True, alpha=0.3)
plt.show()


## Step 4: Multi-output joint surrogate (optional)
SBI's NPE flow learns the full joint density over D-dim outputs natively when
`output_correlation: "full"` is set. The example below uses a 2-output store.


In [ ]:
import json
from pathlib import Path
import numpy as np

store = ROOT / 'tmp/tutorial_multi_output_store'
(store / 'runs').mkdir(parents=True, exist_ok=True)
if not any((store / 'runs').iterdir()):
    rng = np.random.default_rng(1)
    for idx in range(80):
        a = float(rng.uniform(-2, 2))
        b = float(rng.uniform(-1, 1))
        y1 = 1.7 * a - 0.8 * b + 0.2 + float(rng.normal(0, 0.05))
        y2 = -0.5 * a + 1.2 * b - 0.3 + float(rng.normal(0, 0.05))
        run_dir = store / 'runs' / f'run_{idx:03d}'
        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / 'inputs.json').write_text(json.dumps({'a': a, 'b': b}))
        (run_dir / 'outputs.json').write_text(json.dumps({'y1': y1, 'y2': y2}))

# Reuse the multi-output spec but flip the backend to sbi_npe.
multi_payload = json.loads(
    (ROOT / 'examples/surrogates/surrogate.toy.multi_output.json').read_text()
)
multi_payload.update({
    'name': 'toy_multi_output_sbi',
    'backend': 'sbi_npe',
    'backend_config': {
        'output_correlation': 'full',
        'density_estimator': 'maf',
        'max_num_epochs': 80,
        'training_batch_size': 16,
        'summary_samples': 64,
    },
})
multi_path = ROOT / 'tmp/surrogate.toy.sbi_multi.json'
multi_path.write_text(json.dumps(multi_payload, indent=2))
run_cli('surrogate', 'fit', str(multi_path.relative_to(ROOT)))


## Scientific mini-lesson
- SBI shines when likelihoods are hard to write but simulation is available.
- NPE learns an approximate posterior density from simulated data.
- A D-dim density estimator (`maf`/`nsf`) over D outputs is a true joint surrogate.
- Differences from PyMC are expected: approximation families and training dynamics differ.
